# 🧹 Data Preprocessing

Notebook ini berisi tahapan preprocessing yang dilakukan untuk menyiapkan dataset sebelum proses pemodelan machine learning. Tahapan ini bertujuan meningkatkan kualitas data serta membentuk fitur-fitur yang relevan untuk proses prediksi harga komoditas pangan bulanan.

Proses yang dilakukan pada notebook ini meliputi:
- Pembersihan nama kolom dan nilai pada dataset.
- Konversi dan pembersihan data harga menjadi format numerik.
- Penanganan missing value menggunakan interpolasi dan forward/backward fill.
- Encoding variabel bulan menjadi representasi numerik.
- Pembuatan kolom tanggal berdasarkan tahun dan bulan.
- Deteksi outlier menggunakan metode Interquartile Range (IQR).
- Feature engineering berupa lag feature, moving average, perubahan harga, dan target harga bulan berikutnya.
- Encoding variabel komoditas menggunakan One-Hot Encoding.
- Penentuan fitur dan target untuk proses pemodelan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Library berhasil diimport")

Load dataset
df = pd.read_csv('data/raw/1778217759.csv')

###Cleaning nama kolom dan teks

In [ ]:
# Cleaning nama kolom dan data teks pada dataset utama
df.columns = df.columns.str.strip()

df["Komoditas"] = df["Komoditas"].astype(str).str.strip()
df["Bulan"] = df["Bulan"].astype(str).str.strip()

print("Nama kolom setelah cleaning:")
print(df.columns.tolist())

print("Jumlah komoditas setelah cleaning:", df["Komoditas"].nunique())
print("Daftar bulan setelah cleaning:")
print(df["Bulan"].unique())

###Cleaning Harga dengan Vectorized Operation

In [ ]:
# Cleaning harga secara vectorized
df["Harga"] = (
    df["Harga"]
    .astype(str)
    .str.replace("Rp", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("-", "", regex=False)
    .str.strip()
)

# Ubah string kosong menjadi NaN
df["Harga"] = df["Harga"].replace("", np.nan)

# Konversi ke numerik float
df["Harga"] = pd.to_numeric(df["Harga"], errors="coerce").astype(float)

print("Tipe data Harga setelah cleaning:", df["Harga"].dtype)
print("Jumlah missing Harga:", df["Harga"].isnull().sum())

display(df[df["Harga"].isnull()][["Komoditas", "Tahun", "Bulan", "Harga"]].head(30))

###Imputasi Missing Value Harga

In [ ]:
print("Missing Harga sebelum imputasi:", df["Harga"].isna().sum())

df["Harga"] = df.groupby("Komoditas")["Harga"].transform(
    lambda x: x.interpolate(method="linear")
)

df["Harga"] = df.groupby("Komoditas")["Harga"].transform(
    lambda x: x.ffill().bfill()
)

print("Missing Harga setelah imputasi:", df["Harga"].isna().sum())

In [ ]:
display(df[df["Harga"].isnull()][["Komoditas", "Tahun", "Bulan", "Harga"]].head(30))

###Statistik Harga Setelah Cleaning

In [ ]:
display(df["Harga"].describe())

statistik_komoditas = df.groupby("Komoditas")["Harga"].agg(
    Jumlah_Data="count",
    Harga_Min="min",
    Harga_Rata_Rata="mean",
    Harga_Max="max",
    Harga_Std="std"
).reset_index()

display(statistik_komoditas.sort_values("Harga_Rata_Rata", ascending=False))

###Encoding Bulan Langsung pada Kolom Bulan

In [ ]:
bulan_mapping = {
    "Januari": 1,
    "Februari": 2,
    "Maret": 3,
    "April": 4,
    "Mei": 5,
    "Juni": 6,
    "Juli": 7,
    "Agustus": 8,
    "September": 9,
    "Oktober": 10,
    "November": 11,
    "Desember": 12
}

bulan_inverse_mapping = {
    1: "Januari",
    2: "Februari",
    3: "Maret",
    4: "April",
    5: "Mei",
    6: "Juni",
    7: "Juli",
    8: "Agustus",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Desember"
}

df["Bulan"] = df["Bulan"].map(bulan_mapping)

print("Jumlah bulan gagal dikonversi:", df["Bulan"].isna().sum())
print("Daftar bulan setelah encoding:")
print(sorted(df["Bulan"].dropna().unique()))

display(df.head())

###Membuat Kolom Tanggal

In [ ]:
df["Tanggal"] = pd.to_datetime(
    df["Tahun"].astype(str) + "-" + df["Bulan"].astype("Int64").astype(str) + "-01",
    errors="coerce"
)

print("Tanggal tidak valid:", df["Tanggal"].isna().sum())
print("Tanggal awal:", df["Tanggal"].min())
print("Tanggal akhir:", df["Tanggal"].max())

df = df.sort_values(["Komoditas", "Tanggal"]).reset_index(drop=True)

display(df.head())

###Deteksi Outlier Harga

In [ ]:
def detect_outlier_iqr(group):
    q1 = group["Harga"].quantile(0.25)
    q3 = group["Harga"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    group["Outlier"] = (group["Harga"] < lower_bound) | (group["Harga"] > upper_bound)

    return group

df = df.groupby("Komoditas", group_keys=False).apply(detect_outlier_iqr)

print("Jumlah outlier:", df["Outlier"].sum())

display(
    df[df["Outlier"] == True][
        ["Komoditas", "Tahun", "Bulan", "Harga"]
    ].head(30)
)

###Feature Engineering: Lag Harga

In [ ]:
df["Harga_Lag_1"] = df.groupby("Komoditas")["Harga"].shift(1)
df["Harga_Lag_2"] = df.groupby("Komoditas")["Harga"].shift(2)
df["Harga_Lag_3"] = df.groupby("Komoditas")["Harga"].shift(3)

display(
    df[
        [
            "Komoditas",
            "Tanggal",
            "Tahun",
            "Bulan",
            "Harga",
            "Harga_Lag_1",
            "Harga_Lag_2",
            "Harga_Lag_3"
        ]
    ].head(15)
)

###Feature Engineering: Moving Average

In [ ]:
df["Moving_Average_3"] = df.groupby("Komoditas")["Harga"].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

df["Moving_Average_6"] = df.groupby("Komoditas")["Harga"].transform(
    lambda x: x.shift(1).rolling(window=6).mean()
)

display(
    df[
        [
            "Komoditas",
            "Tanggal",
            "Harga",
            "Moving_Average_3",
            "Moving_Average_6"
        ]
    ].head(15)
)

###Feature Engineering: Perubahan Harga

In [ ]:
df["Perubahan_Harga"] = df["Harga"] - df["Harga_Lag_1"]

df["Persentase_Perubahan_Harga"] = (
    df["Perubahan_Harga"] / df["Harga_Lag_1"]
) * 100

display(
    df[
        [
            "Komoditas",
            "Tanggal",
            "Harga",
            "Perubahan_Harga",
            "Persentase_Perubahan_Harga"
        ]
    ].head(15)
)

###Membuat Target Prediksi

In [ ]:
df["Target_Harga_Bulan_Berikutnya"] = df.groupby("Komoditas")["Harga"].shift(-1)

display(
    df[
        [
            "Komoditas",
            "Tanggal",
            "Harga",
            "Target_Harga_Bulan_Berikutnya"
        ]
    ].head(15)
)

###Drop Missing dari Fitur dan Target

In [ ]:
fitur_wajib = [
    "Harga_Lag_1",
    "Harga_Lag_2",
    "Harga_Lag_3",
    "Moving_Average_3",
    "Moving_Average_6",
    "Perubahan_Harga",
    "Persentase_Perubahan_Harga",
    "Target_Harga_Bulan_Berikutnya"
]

print("Ukuran sebelum drop missing:", df.shape)

df_model = df.dropna(subset=fitur_wajib).copy()

print("Ukuran setelah drop missing:", df_model.shape)

display(df_model.head())

###Encoding Komoditas

In [ ]:
df_model_encoded = pd.get_dummies(
    df_model,
    columns=["Komoditas"],
    drop_first=False
)

print("Ukuran data setelah encoding:", df_model_encoded.shape)

display(df_model_encoded.head())

###Menentukan Fitur dan Target

In [ ]:
target = "Target_Harga_Bulan_Berikutnya"

fitur_numerik = [
    "Tahun",
    "Bulan",
    "Harga",
    "Harga_Lag_1",
    "Harga_Lag_2",
    "Harga_Lag_3",
    "Moving_Average_3",
    "Moving_Average_6",
    "Perubahan_Harga",
    "Persentase_Perubahan_Harga"
]

fitur_komoditas = [
    col for col in df_model_encoded.columns
    if col.startswith("Komoditas_")
]

fitur_final = fitur_numerik + fitur_komoditas

X = df_model_encoded[fitur_final]
y = df_model_encoded[target]

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)

display(X.head())
display(y.head())

###Dynamic Time-Based Split

In [ ]:
df_model_encoded = df_model_encoded.sort_values("Tanggal").reset_index(drop=True)

# Ambil tanggal unik
unique_dates = sorted(df_model_encoded["Tanggal"].unique())

# Tentukan titik split 80% berdasarkan tanggal unik
split_index = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_index]

train_data = df_model_encoded[df_model_encoded["Tanggal"] < split_date]
test_data = df_model_encoded[df_model_encoded["Tanggal"] >= split_date]

X_train = train_data[fitur_final]
y_train = train_data[target]

X_test = test_data[fitur_final]
y_test = test_data[target]

print("Split date:", split_date)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPeriode train:")
print(train_data["Tanggal"].min(), "sampai", train_data["Tanggal"].max())

print("\nPeriode test:")
print(test_data["Tanggal"].min(), "sampai", test_data["Tanggal"].max())

###Scaling Fitur Numerik

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[fitur_numerik] = scaler.fit_transform(X_train[fitur_numerik])
X_test_scaled[fitur_numerik] = scaler.transform(X_test[fitur_numerik])

display(X_train_scaled.head())